# Heatsink Parametric Study — Square-Wave Fin Parametrization

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd().resolve()
while repo_root.name and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

example_dir = str(repo_root / "examples" / "heatflow" / "heatsink")
if example_dir not in sys.path:
    sys.path.insert(0, example_dir)

import heatsink_parametric as hp
import heatsink_tutorial as ht
import matplotlib.pyplot as plt
import pandas as pd
%matplotlib inline

from py2femm.client import FemmClient

assert ht.server_is_healthy(), "py2femm server not running on localhost:8082"
print("Server OK")

## 1. Parameter Grid

In [ ]:
configs = hp.build_sweep_grid()
print(f"Full factorial: {len(hp.L_VALUES)}x{len(hp.PITCH_RATIOS)}"
      f"x{len(hp.DUTY_CYCLES)}x{len(hp.BASE_RATIOS)} = "
      f"{len(hp.L_VALUES)*len(hp.PITCH_RATIOS)*len(hp.DUTY_CYCLES)*len(hp.BASE_RATIOS)}")
print(f"Valid configs:  {len(configs)}")

preview = pd.DataFrame([
    {"L": c.base_width, "pitch/L": c.pitch_actual/c.base_width,
     "D": c.duty_cycle, "r_b": c.base_ratio,
     "n": c.n_fins, "w_f": c.fin_width, "g": c.gap}
    for c in configs[:10]
])
display(preview)

## 2. Run Sweep (~10 min)

In [ ]:
client = FemmClient(mode="remote", url="http://localhost:8082")
df = hp.run_sweep(configs, client)
print(f"\nCompleted: {len(df)}/{len(configs)} configs")
df.to_csv("heatsink_parametric_results.csv", index=False)
print("Results saved to heatsink_parametric_results.csv")

## 3. Results Table

In [ ]:
display(df.sort_values("R_th").head(20).style.format({
    "R_th": "{:.3f}", "T_avg": "{:.1f}", "T_max": "{:.1f}", "T_min": "{:.1f}",
    "A_cross": "{:.0f}", "R_th_per_area": "{:.6f}",
}))

## 4. 1D Sensitivity Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for ax, param in zip(axes.flat, ["base_width", "pitch_ratio", "duty_cycle", "base_ratio"]):
    hp.plot_sensitivity(df, param, ax=ax)
plt.suptitle("1D Sensitivity - R_th and A_cross", fontsize=13)
plt.tight_layout()
plt.show()

## 5. 2D Heatmaps

In [ ]:
hp.plot_heatmap(df)
plt.show()

## 6. Scaling: R_th vs Base Width

In [ ]:
hp.plot_scaling(df)
plt.show()

## 7. Contact Mode Comparison

In [ ]:
top5_rows = df.nsmallest(5, "R_th")
top5_configs = [
    hp.HeatsinkConfig(
        base_width=row.base_width, pitch=row.pitch_ratio * row.base_width,
        duty_cycle=row.duty_cycle, base_ratio=row.base_ratio,
    )
    for _, row in top5_rows.iterrows()
]

contact_results = {}
for mode in ["centered", "single_fin", "between_fins"]:
    mode_configs = [
        hp.HeatsinkConfig(
            base_width=c.base_width, pitch=c.pitch, duty_cycle=c.duty_cycle,
            base_ratio=c.base_ratio, contact_mode=mode,
        )
        for c in top5_configs
    ]
    print(f"\n--- Contact mode: {mode} ---")
    contact_results[mode] = hp.run_sweep(mode_configs, client)

hp.plot_contact_comparison(contact_results)
plt.show()

## 8. Best Configurations

In [ ]:
best3 = df.nsmallest(3, "R_th")
print("=== Top 3 Configurations ===\n")
for i, (_, row) in enumerate(best3.iterrows()):
    print(f"#{i+1}: L={row.base_width:.0f}mm, pitch/L={row.pitch_ratio:.2f}, "
          f"D={row.duty_cycle:.2f}, r_b={row.base_ratio:.2f}")
    print(f"     n={row.n_fins:.0f} fins, w_f={row.fin_width:.1f}mm, gap={row.gap:.1f}mm")
    print(f"     R_th={row.R_th:.3f} K/W, A_cross={row.A_cross:.0f} mm^2")
    print()

top3_configs = [
    hp.HeatsinkConfig(
        base_width=row.base_width, pitch=row.pitch_ratio * row.base_width,
        duty_cycle=row.duty_cycle, base_ratio=row.base_ratio,
    )
    for _, row in best3.iterrows()
]
labels = [
    f"#{i+1}: L={c.base_width:.0f}, R_th={r:.3f}"
    for i, (c, r) in enumerate(zip(top3_configs, best3["R_th"]))
]
hp.plot_geometry_overlay(top3_configs, labels=labels)
plt.show()